In [ ]:
!pip install pandas nltk
!pip install scikit-learn gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 40.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download resource NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

# ===============================
# LOAD DATASET
# ===============================
df = pd.read_csv("/content/imdb-movies-dataset.csv", encoding='utf-8', on_bad_lines='warn')

print("Jumlah data awal:", df.shape)
print("Kolom dataset:", df.columns)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


FileNotFoundError: [Errno 2] No such file or directory: '/content/imdb-movies-dataset.csv'

Dataset terdiri dari 10.000 data film dengan 15 kolom atribut, seperti judul, genre, rating, director, description, dan review. Jumlah data udah memenuhi requirement minimal dataset (>10.000 data) buat proses klasifikasi genre film.

# PREPOCESSING

In [ ]:
# ===============================
# TEXT PREPROCESSING
# ===============================

def preprocess_text(text):
    # Handle missing value
    text = str(text)

    # Hapus karakter selain huruf
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Case folding + tokenization
    tokens = word_tokenize(text.lower())

    # Stopword removal
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]

    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return tokens

In [ ]:
# Contoh preprocessing 1 data
sample_text = df["Description"][0]
preprocessed = preprocess_text(sample_text)

print("\nContoh preprocessing:")
print("Sebelum:", sample_text)
print("Sesudah:", preprocessed)

Tahap preprocessing berhasil membersihkan teks synopsis dengan mengubah huruf jadi lowercase, hapus tanda baca dan angka, melakukan tokenisasi, hapus stopword, serta melakukan lemmatization sehingga teksnya jadi lebih bersih dan siap digunakan pada tahap feature extraction.

In [ ]:
# Terapkan ke seluruh dataset
df['preprocessed'] = df['Description'].fillna("").apply(preprocess_text)

print("\nHasil 5 data pertama:")
print(df[["Description", "preprocessed"]].head(5))

In [ ]:
# ===============================
# LABEL PREPARATION
# ===============================
from sklearn.preprocessing import MultiLabelBinarizer

# Handle missing value pada genre
df['Genre'] = df['Genre'].fillna('')

# Ubah string genre menjadi list
df['Genre_list'] = df['Genre'].apply(
    lambda x: x.split(', ') if x != '' else []
)

# Multi-label encoding
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['Genre_list'])

print("\nJumlah kelas genre:", len(mlb.classes_))
print("Shape label:", y.shape)
print("Daftar genre:", mlb.classes_)

Dataset punya 23 genre unik. Karena satu film dapat punya lebih dari satu genre, jadi digunakan MultiLabelBinarizer untuk mengubah label genre menjadi format numerik multi-label dengan shape (10000, 23).

# FEATURE EXTRACTION

In [ ]:
# ===============================
# TF-IDF
# ===============================

from sklearn.feature_extraction.text import TfidfVectorizer

# Menggabungkan list token menjadi string utuh untuk TF-IDF
df['preprocessed_string'] = df['preprocessed'].apply(lambda x: ' '.join(x))

# Inisialisasi model TF-IDF
# max_features=5000 digunakan agar dimensi tidak terlalu besar dan komputasi model lebih ringan
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Melakukan ekstraksi fitur
X_tfidf = tfidf_vectorizer.fit_transform(df['preprocessed_string'])

print("Dimensi TF-IDF:", X_tfidf.shape)

TF-IDF dipakai untuk merepresentasikan teks berdasarkan frekuensi kata dan tingkat kepentingannya dalam dokumen. Hasil transformasi hasilnya matriks dengan dimensi (10000, 5000), yang berarti terdapat 10.000 dokumen dan 5.000 fitur kata.

In [ ]:
# ===============================
# Word2Vec
# ===============================

import numpy as np
from gensim.models import Word2Vec

# Menyiapkan corpus data dalam bentuk list of lists
vector_size = 100
sentences = df['preprocessed'].tolist()

# Melatih model Word2Vec
# vector_size=100 (ukuran dimensi vektor), window=5 (jarak konteks kata), min_count=2 (minimal muncul 2 kali)
w2v_model = Word2Vec(sentences, vector_size=vector_size, window=5, min_count=2, workers=4)

# Fungsi untuk mengambil rata-rata vektor dari kata-kata dalam satu dokumen
def get_document_vector(tokens, model, vector_size):
    # Hanya ambil kata yang dikenali oleh vocabulary model
    valid_words = [word for word in tokens if word in model.wv.key_to_index]
    if valid_words:
        # Hitung rata-rata vektor dari semua kata yang valid
        return np.mean(model.wv[valid_words], axis=0)
    else:
        # Jika dokumen kosong/kata tidak ada yang dikenali, kembalikan array nol
        return np.zeros(vector_size)

# Terapkan fungsi ke seluruh data sinopsis
X_w2v = np.array([get_document_vector(tokens, w2v_model, vector_size) for tokens in sentences])

print("Dimensi Word2Vec:", X_w2v.shape)

Word2Vec dipakai untuk menghasilkan representasi vektor kata berbasis konteks. Vektor dokumen diperoleh dengan menghitung rata-rata vektor kata dalam setiap dokumen, sehingga menghasilkan dimensi (10000, 100).

In [ ]:
# ===============================
# FastText
# ===============================

from gensim.models import FastText

# Melatih model FastText
ft_model = FastText(sentences, vector_size=vector_size, window=5, min_count=2, workers=4)

# Terapkan fungsi rata-rata vektor (get_document_vector sudah didefinisikan di cell sebelumnya)
X_fasttext = np.array([get_document_vector(tokens, ft_model, vector_size) for tokens in sentences])

print("Dimensi FastText:", X_fasttext.shape)

FastText dipakai untuk menghasilkan embedding berbasis subword, sehingga mampu menangani kata yang jarang muncul. Representasi dokumen juga diperoleh melalui rata-rata vektor kata, dengan dimensi (10000, 100).

# MODELING

In [ ]:
# ===============================
# Split Data
# ===============================

from sklearn.model_selection import train_test_split

# Membagi data menjadi data training (80%) dan testing (20%)
# Tujuan: agar model bisa belajar dari data train dan diuji di data baru (test)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# Menampilkan ukuran data
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


Dataset dibagi jadi data latih dan data uji dengan rasio 80:20. Data latih digunakan untuk melatih model, sedangkan data uji digunakan untuk mengevaluasi performa model pada data yang belum pernah dilihat sebelumnya.

In [ ]:
# ===============================
# Naive Bayes
# ===============================

from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier

# Menggunakan OneVsRest karena ini adalah multi-label classification
# Naive Bayes cocok buat data berbasis teks kaya TF-IDF
nb_model = OneVsRestClassifier(MultinomialNB())

# Melatih model menggunakan data training
nb_model.fit(X_train, y_train)

# Melakukan prediksi pada data test
y_pred_nb = nb_model.predict(X_test)

Naive Bayes dipakai karena efektif untuk klasifikasi teks. Modelnya bekerja berdasarkan probabilitas kemunculan kata dalam setiap kelas. Karena dataset bersifat multi-label, digunakan pendekatan One-vs-Rest.

In [ ]:
# ===============================
# Logistic Regression
# ===============================

from sklearn.linear_model import LogisticRegression

# Logistic Regression digunakan untuk memodelkan hubungan antara fitur dan label
# max_iter diperbesar agar model konvergen (ga error)
lr_model = OneVsRestClassifier(LogisticRegression(max_iter=1000))

# Training model
lr_model.fit(X_train, y_train)

# Prediksi
y_pred_lr = lr_model.predict(X_test)

Logistic Regression dipakai untuk memodelkan probabilitas suatu data termasuk ke dalam suatu kelas. Modelnya bekerja dengan mencari batas pemisah antar kelas dan cocok untuk klasifikasi multi-label dengan pendekatan One-vs-Rest.

In [ ]:
# ===============================
# SVM
# ===============================

from sklearn.svm import LinearSVC

# SVM dipake untuk mencari hyperplane terbaik yang memisahkan kelas
svm_model = OneVsRestClassifier(LinearSVC())

# Training model
svm_model.fit(X_train, y_train)

# Prediksi
y_pred_svm = svm_model.predict(X_test)

Support Vector Machine (SVM) dipake untuk mencari batas pemisah terbaik antar kelas. SVM sangat efektif buat data dengan dimensi tinggi seperti TF-IDF dan sering memberikan hasil yang baik pada klasifikasi teks.

In [ ]:
# ===============================
# Evaluation
# ===============================

from sklearn.metrics import classification_report

# Menampilkan hasil evaluasi model Naive Bayes
print("=== Naive Bayes ===")
print(classification_report(y_test, y_pred_nb, zero_division=0))

# Logistic Regression
print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, zero_division=0))

# SVM
print("=== SVM ===")
print(classification_report(y_test, y_pred_svm, zero_division=0))

**Analisis per model**

1. Naive Bayes: kurang mampu menangkap kompleksitas hubungan antar kata dalam teks, makanya performanya relatif rendah terutama pada kelas minoritas.
2. Logistic Regression: menunjukkan peningkatan performa dibanding Naive Bayes, cuman masih kesulitan dalam memprediksi kelas dengan jumlah data yang sedikit.
3. SVM: memberikan performa terbaik karena kemampuannya dalam menangani data berdimensi tinggi seperti TF-IDF dan menghasilkan batas pemisah yang optimal antar kelas.

In [ ]:
# ===============================
# 1. Ulang Modeling Word2Vec
# ===============================

# Split Data #
from sklearn.model_selection import train_test_split

# Membagi data Word2Vec
X_train, X_test, y_train, y_test = train_test_split(
    X_w2v,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Word2Vec - X_train:", X_train.shape)
print("Word2Vec - X_test:", X_test.shape)

In [ ]:
# Naive Bayes #
from sklearn.naive_bayes import GaussianNB
from sklearn.multiclass import OneVsRestClassifier

# NB untuk data numerik (Word2Vec)
nb_w2v = OneVsRestClassifier(GaussianNB())

nb_w2v.fit(X_train, y_train)
y_pred_nb_w2v = nb_w2v.predict(X_test)

In [ ]:
# Logistic Regression #
from sklearn.linear_model import LogisticRegression

lr_w2v = OneVsRestClassifier(LogisticRegression(max_iter=1000))

lr_w2v.fit(X_train, y_train)
y_pred_lr_w2v = lr_w2v.predict(X_test)


In [ ]:
# SVM #

from sklearn.svm import LinearSVC

svm_w2v = OneVsRestClassifier(LinearSVC())

svm_w2v.fit(X_train, y_train)
y_pred_svm_w2v = svm_w2v.predict(X_test)

In [ ]:
# Evaluation #
from sklearn.metrics import classification_report

print("=== W2V - Naive Bayes ===")
print(classification_report(y_test, y_pred_nb_w2v, zero_division=0))

print("=== W2V - Logistic Regression ===")
print(classification_report(y_test, y_pred_lr_w2v, zero_division=0))

print("=== W2V - SVM ===")
print(classification_report(y_test, y_pred_svm_w2v, zero_division=0))

In [ ]:
# ===============================
# 1. Ulang Modeling FastText
# ===============================

# Split Data #
X_train, X_test, y_train, y_test = train_test_split(
    X_fasttext,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("FastText - X_train:", X_train.shape)
print("FastText - X_test:", X_test.shape)

In [ ]:
# Naive Bayes #
nb_ft = OneVsRestClassifier(GaussianNB())

nb_ft.fit(X_train, y_train)
y_pred_nb_ft = nb_ft.predict(X_test)

In [ ]:
# Logistic Regression #
lr_ft = OneVsRestClassifier(LogisticRegression(max_iter=1000))

lr_ft.fit(X_train, y_train)
y_pred_lr_ft = lr_ft.predict(X_test)

In [ ]:
# SVM #
svm_ft = OneVsRestClassifier(LinearSVC())

svm_ft.fit(X_train, y_train)
y_pred_svm_ft = svm_ft.predict(X_test)

In [ ]:
# Evaluation #
print("=== FastText - Naive Bayes ===")
print(classification_report(y_test, y_pred_nb_ft, zero_division=0))

print("=== FastText - Logistic Regression ===")
print(classification_report(y_test, y_pred_lr_ft, zero_division=0))

print("=== FastText - SVM ===")
print(classification_report(y_test, y_pred_svm_ft, zero_division=0))

#  Comparison Model

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Buat list untuk menampung semua hasil
comparison_results = []

# Re-run train-test split for each feature to ensure distinct test sets
# For TF-IDF
X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, shuffle=True
)

# For Word2Vec
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    X_w2v, y, test_size=0.2, random_state=42, shuffle=True
)

# For FastText
X_train_ft, X_test_ft, y_train_ft, y_test_ft = train_test_split(
    X_fasttext, y, test_size=0.2, random_state=42, shuffle=True
)

# Structure to hold feature sets and their corresponding models
all_comparisons = {
    'TF-IDF': {
        'X_test_data': X_test_tfidf,
        'y_true_data': y_test_tfidf,
        'models': {
            'Naive Bayes': nb_model,
            'Logistic Regression': lr_model,
            'SVM': svm_model
        }
    },
    'Word2Vec': {
        'X_test_data': X_test_w2v,
        'y_true_data': y_test_w2v,
        'models': {
            'Naive Bayes': nb_w2v,
            'Logistic Regression': lr_w2v,
            'SVM': svm_w2v
        }
    },
    'FastText': {
        'X_test_data': X_test_ft,
        'y_true_data': y_test_ft,
        'models': {
            'Naive Bayes': nb_ft,
            'Logistic Regression': lr_ft,
            'SVM': svm_ft
        }
    }
}

# 3. Loop untuk menghitung metrik secara otomatis
for feat_name, feat_data in all_comparisons.items():
    X_test_data = feat_data['X_test_data']
    y_true = feat_data['y_true_data']

    for model_name, model_obj in feat_data['models'].items():
        y_pred = model_obj.predict(X_test_data)

        # Karena ini Multi-label, kita gunakan rata-rata 'micro'
        comparison_results.append({
            'Feature': feat_name,
            'Model': model_name,
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, average='micro'),
            'Recall': recall_score(y_true, y_pred, average='micro'),
            'F1-Score': f1_score(y_true, y_pred, average='micro')
        })
# 4. Tampilkan tabel perbandingan
df_compare = pd.DataFrame(comparison_results)
print("=== TABEL PERBANDINGAN MODEL) ===")
display(df_compare.sort_values(by='F1-Score', ascending=False))

=== TABEL PERBANDINGAN MODEL) ===


,Feature,Model,Accuracy,Precision,Recall,F1-Score
2,TF-IDF,SVM,0.0865,0.628359,0.437413,0.515781
1,TF-IDF,Logistic Regression,0.0705,0.721888,0.310448,0.434178
0,TF-IDF,Naive Bayes,0.0670,0.738773,0.268458,0.393811
7,FastText,Logistic Regression,0.0430,0.537426,0.198607,0.290032
4,Word2Vec,Logistic Regression,0.0460,0.576239,0.180498,0.274890
8,FastText,SVM,0.0430,0.569132,0.176119,0.268997
5,Word2Vec,SVM,0.0430,0.619534,0.169154,0.265750
6,FastText,Naive Bayes,0.0000,0.141327,0.496318,0.220007
3,Word2Vec,Naive Bayes,0.0000,0.130488,0.485572,0.205699


In [36]:
# --- SAVE MODEL---
import joblib
import os

os.makedirs('model', exist_ok=True)

joblib.dump(svm_model, 'model/svm_model.pkl')
joblib.dump(tfidf_vectorizer, 'model/tfidf_vectorizer.pkl')
joblib.dump(mlb, 'model/mlb.pkl')

print("✅ Model berhasil disimpan!")

✅ Model berhasil disimpan!


In [39]:
from google.colab import files

files.download('model/svm_model.pkl')
files.download('model/tfidf_vectorizer.pkl')
files.download('model/mlb.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Simple Deployment Menggunakan Gradio

In [ ]:
import gradio as gr
import traceback

def predict_genre_english(description):
    try:
        # Validasi input
        if not description.strip():
            return "❌ Input tidak boleh kosong."

        # Preprocessing
        clean_tokens = preprocess_text(description)
        clean_text = ' '.join(clean_tokens)

        # TF-IDF transform (tanpa fit ulang)
        vec = tfidf_vectorizer.transform([clean_text])

        # Prediksi
        pred = svm_model.predict(vec)

        # Decode label
        genres = mlb.inverse_transform(pred)

        # Format output
        if genres and len(genres[0]) > 0:
            hasil_genre = ", ".join(genres[0])
        else:
            hasil_genre = "Genre tidak terdeteksi"

        return f"✅ Success!\n\nPredicted Genre: {hasil_genre}"

    except Exception:
        error_msg = traceback.format_exc()
        return f"❌ ERROR DETECTED!\n\n{error_msg}"


# UI Gradio
demo = gr.Interface(
    fn=predict_genre_english,
    inputs=gr.Textbox(
        label="Input Movie Description (English)",
        placeholder="Example: A brave warrior fights for justice in a fantasy world...",
        lines=4
    ),
    outputs=gr.Textbox(label="Prediction Result", lines=6),
    title="🎬 Movie Genre Predictor",
    description="Aplikasi ini memprediksi genre film berdasarkan deskripsi menggunakan model SVM + TF-IDF.",
    theme="soft"
)

demo.launch(share=True)

In [31]:
import gradio as gr
import traceback

def predict_genre_english(description):
    try:
        if not description.strip():
            return "❌ Input tidak boleh kosong."

        clean_tokens = preprocess_text(description)
        clean_text = ' '.join(clean_tokens)

        vec = tfidf_vectorizer.transform([clean_text])
        pred = svm_model.predict(vec)

        genres = mlb.inverse_transform(pred)

        if genres and len(genres[0]) > 0:
            hasil_genre = ", ".join(genres[0])
        else:
            hasil_genre = "Genre tidak terdeteksi"

        return f"✅ Success!\n\nPredicted Genre: {hasil_genre}"

    except Exception as e:
        return f"❌ Error: {str(e)}"

# Membuat UI Gradio
demo = gr.Interface(
    fn=predict_genre_english,
    inputs=gr.Textbox(
        label="Input Movie Description (English)",
        placeholder="Example: A brave warrior fights for justice in a fantasy world...",
        lines=4
    ),
    outputs=gr.Textbox(label="Prediction Result / Error Details", lines=10),
    title="🎬 Movie Genre Predictor",
    description="Aplikasi ini memprediksi genre film berdasarkan deskripsinya menggunakan NLP. (Input bahasa Inggris saja)",
    theme="soft"
)

# Launch aplikasi
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b28a9ecccee8c7474a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
